# Telecom Customer Churn Prediction Using Data Analytics and AI

**AICTE | IBM SkillsBuild Data Analytics with AI Internship 2026**

**Student:** Anubhab Nandi  
**Institution:** Guru Nanak Institute of Technology  
**Program:** BCA

## Project Overview

This project analyzes telecom customer information and uses machine learning to predict customer churn.

The project uses the public **Cell2Cell Telecom Customer Churn dataset**, which contains more than 50,000 customer records and 58 variables in the referenced training file. The dataset is substantially larger than the 10,000-customer minimum requested for this project.

The workflow includes data loading, data quality checking, preprocessing, exploratory data analysis, visualization, machine learning, model evaluation, and feature-importance analysis.

## AI / Machine Learning

A Random Forest Classifier is used for binary customer churn prediction. AI-assisted development tools can be used during development for code assistance, debugging, explanation, and documentation. The final notebook contains the complete implemented workflow.

## Dataset Source

Public dataset repository:
https://github.com/leylatulu/Telecom-Churn-Analysis

The repository describes the Cell2Cell training data as 51,047 samples with 58 features.

Dataset file used:
`cell2celltrain.csv`


In [ ]:
# Install required libraries if they are not already installed
%pip install pandas numpy matplotlib scikit-learn -q


In [ ]:
# Import libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

print("Libraries imported successfully.")


In [ ]:
# Load the public Cell2Cell training dataset

DATASET_URL = (
    "https://raw.githubusercontent.com/leylatulu/"
    "Telecom-Churn-Analysis/main/cell2celltrain.csv"
)

df = pd.read_csv(DATASET_URL)

print("Dataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

if df.shape[0] >= 10000:
    print("Dataset requirement check: PASSED - more than 10,000 customer records.")
else:
    print("Dataset requirement check: FAILED - fewer than 10,000 records.")

df.head()


In [ ]:
# Basic dataset inspection

print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes.value_counts())

print("\nFirst five rows:")
display(df.head())

print("\nMissing values - top 20 columns:")
display(df.isnull().sum().sort_values(ascending=False).head(20))


In [ ]:
# Identify the churn target column automatically

possible_targets = [
    c for c in df.columns
    if "churn" in str(c).lower()
]

print("Possible churn columns:", possible_targets)

if not possible_targets:
    raise ValueError(
        "No churn target column was found. Please inspect the dataset columns."
    )

TARGET = possible_targets[0]
print("Selected target column:", TARGET)

print("\nTarget distribution:")
print(df[TARGET].value_counts(dropna=False))


In [ ]:
# Clean the target column

# Remove rows where the target is missing.
df = df.dropna(subset=[TARGET]).copy()

# Convert common churn labels into 0/1.
def encode_target(series):
    if pd.api.types.is_numeric_dtype(series):
        unique_values = sorted(series.dropna().unique().tolist())
        if set(unique_values).issubset({0, 1}):
            return series.astype(int)

    text = series.astype(str).str.strip().str.lower()

    positive = {"yes", "y", "true", "1", "churn", "churned"}
    negative = {"no", "n", "false", "0", "stay", "stayed", "false."}

    mapped = text.map(
        lambda x: 1 if x in positive else (0 if x in negative else np.nan)
    )

    if mapped.isna().any():
        # Fallback: factorize a binary target if labels are unusual.
        unique = text.dropna().unique()
        if len(unique) == 2:
            mapping = {unique[0]: 0, unique[1]: 1}
            mapped = text.map(mapping)
        else:
            raise ValueError(
                "The target is not binary or its labels could not be recognized."
            )

    return mapped.astype(int)

y = encode_target(df[TARGET])

print("Encoded target distribution:")
print(y.value_counts())


In [ ]:
# Separate features from target

X_raw = df.drop(columns=[TARGET]).copy()

# Remove obvious identifier columns because they do not provide useful
# predictive information and may cause data leakage.
identifier_keywords = [
    "customerid",
    "customer_id",
    "subscriberid",
    "subscriber_id",
    "accountid",
    "account_id",
    "phone",
    "mobile"
]

drop_columns = []

for column in X_raw.columns:
    name = str(column).lower().replace(" ", "").replace("-", "").replace(".", "")
    if any(keyword in name for keyword in identifier_keywords):
        drop_columns.append(column)

X_raw = X_raw.drop(columns=drop_columns, errors="ignore")

print("Removed identifier-like columns:", drop_columns)
print("Remaining features:", X_raw.shape[1])


In [ ]:
# Data preprocessing

# Separate numeric and categorical columns.
numeric_columns = X_raw.select_dtypes(include=np.number).columns.tolist()
categorical_columns = X_raw.select_dtypes(exclude=np.number).columns.tolist()

# Fill missing numeric values with median.
for column in numeric_columns:
    X_raw[column] = X_raw[column].fillna(X_raw[column].median())

# Fill missing categorical values with the most frequent value.
for column in categorical_columns:
    mode = X_raw[column].mode()
    fill_value = mode.iloc[0] if not mode.empty else "Unknown"
    X_raw[column] = X_raw[column].fillna(fill_value).astype(str)

# One-hot encode categorical variables.
X = pd.get_dummies(X_raw, drop_first=True)

# Convert boolean columns to integers.
X = X.astype(float)

print("Preprocessing completed.")
print("Final feature matrix shape:", X.shape)


In [ ]:
# Exploratory Data Analysis: churn distribution

target_counts = y.value_counts().sort_index()

labels = ["Stayed", "Churned"]

plt.figure(figsize=(7, 5))
plt.bar(labels, [target_counts.get(0, 0), target_counts.get(1, 0)])
plt.title("Customer Churn Distribution")
plt.xlabel("Customer Status")
plt.ylabel("Number of Customers")
plt.tight_layout()
plt.show()

print("Stayed:", target_counts.get(0, 0))
print("Churned:", target_counts.get(1, 0))
print("Churn rate:", f"{y.mean() * 100:.2f}%")


In [ ]:
# Explore numeric feature distributions

if numeric_columns:
    selected_numeric = numeric_columns[:6]

    for column in selected_numeric:
        plt.figure(figsize=(7, 4))
        plt.hist(X_raw[column], bins=30)
        plt.title(f"Distribution of {column}")
        plt.xlabel(column)
        plt.ylabel("Number of Customers")
        plt.tight_layout()
        plt.show()
else:
    print("No numeric columns were available for histogram analysis.")


In [ ]:
# Correlation analysis for numeric variables

if len(numeric_columns) >= 2:
    correlation_df = X_raw[numeric_columns].corr()

    plt.figure(figsize=(10, 7))
    plt.imshow(correlation_df, aspect="auto")
    plt.colorbar()
    plt.title("Correlation Matrix of Numeric Features")
    plt.xticks(
        range(len(correlation_df.columns)),
        correlation_df.columns,
        rotation=90,
        fontsize=7
    )
    plt.yticks(
        range(len(correlation_df.columns)),
        correlation_df.columns,
        fontsize=7
    )
    plt.tight_layout()
    plt.show()
else:
    print("Not enough numeric variables for correlation analysis.")


In [ ]:
# Train-test split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])


In [ ]:
# Scale numeric model inputs

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed.")


In [ ]:
# Train Random Forest Classifier

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

model.fit(X_train_scaled, y_train)

print("Random Forest model trained successfully.")


In [ ]:
# Generate predictions and evaluate the model

y_pred = model.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Score": [accuracy, precision, recall, f1]
})

metrics["Score"] = metrics["Score"].round(4)

display(metrics)

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Stayed", "Churned"],
    zero_division=0
))


In [ ]:
# Confusion matrix

cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Stayed", "Churned"]
)

disp.plot()
plt.title("Customer Churn Confusion Matrix")
plt.show()


In [ ]:
# Feature importance

feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
}).sort_values(
    by="Importance",
    ascending=False
)

print("Top 20 features:")
display(feature_importance.head(20))


In [ ]:
# Visualize top 15 important features

top_features = feature_importance.head(15).sort_values("Importance")

plt.figure(figsize=(10, 7))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.title("Top 15 Features for Churn Prediction")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


In [ ]:
# Example prediction for an unseen customer

sample = X_test.iloc[[0]]
sample_scaled = scaler.transform(sample)

prediction = model.predict(sample_scaled)[0]
churn_probability = model.predict_proba(sample_scaled)[0][1]

print("Prediction:")
print("Customer is likely to CHURN." if prediction == 1 else "Customer is likely to STAY.")
print(f"Estimated churn probability: {churn_probability * 100:.2f}%")


In [ ]:
# Final project summary

print("=" * 60)
print("CUSTOMER CHURN PREDICTION PROJECT - FINAL SUMMARY")
print("=" * 60)
print(f"Dataset records used: {len(df):,}")
print(f"Original dataset features: {df.shape[1]}")
print(f"Model features after preprocessing: {X.shape[1]}")
print(f"Training records: {len(X_train):,}")
print(f"Testing records: {len(X_test):,}")
print(f"Algorithm: Random Forest Classifier")
print(f"Accuracy: {accuracy * 100:.2f}%")
print(f"Precision: {precision * 100:.2f}%")
print(f"Recall: {recall * 100:.2f}%")
print(f"F1 Score: {f1 * 100:.2f}%")
print("=" * 60)


## Conclusion

This project demonstrates a complete Data Analytics and Machine Learning workflow for telecom customer churn prediction.

The dataset contains more than 10,000 customer records, allowing the project to analyze a substantially larger customer population than the minimum requested for the project.

The analysis includes data quality checking, preprocessing, exploratory data analysis, visualization, feature transformation, model training, evaluation, and feature-importance analysis.

The Random Forest model uses historical customer attributes to classify customers into churn and non-churn groups. The evaluation metrics provide a quantitative assessment of model performance.

### Future Improvements

- Compare Random Forest with Logistic Regression, Decision Tree, and Gradient Boosting.
- Perform hyperparameter tuning.
- Use cross-validation.
- Apply advanced explainability techniques such as SHAP.
- Develop an interactive dashboard.
- Deploy the trained model as a web application.

### Important Note

The dataset is a public educational/research dataset. Model predictions are analytical estimates and should not be treated as guaranteed future outcomes.

### AI-Assisted Development

AI-assisted development tools, including IBM Bob where applicable, may be used for code assistance, debugging, explanation, and documentation during development. The student should review and understand the submitted notebook and ensure that the final project accurately represents the work submitted.
